# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a detailed walkthrough for loading and exploring the FAIR² Mental Health Survey dataset from Kilifi County, Kenya, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Croissant datasets organize records into record sets. Each record set, field, and column is uniquely addressed by its `@id`.

Let's examine the structure:

In [ ]:
# List all record sets and their @ids
print("Available record sets:")
record_sets = []
for recordset in dataset.metadata.record_sets:
    print(f"- {recordset['@id']} (name: {recordset.get('name', '')})")
    record_sets.append(recordset['@id'])

# For each record set, list fields and columns
for recordset in dataset.metadata.record_sets:
    print(f"\nRecord set @id: {recordset['@id']} (name: {recordset.get('name', '')})")
    fields = recordset.get('fields', [])
    columns = recordset.get('columns', [])
    print("Fields:")
    for field in fields:
        print(f"  - {field['@id']}: {field.get('name', '')}")
    print("Columns:")
    for column in columns:
        print(f"  - {column['@id']}: {column.get('name', '')}")

### List Example Records from the Main Record Set

Let's print the first few records from the principal record set for inspection.

In Croissant, entities are referenced by their `@id`; below we use the main record set's `@id`.

In [ ]:
# Let's get the @id of the principal record set (assumed first)
main_record_set_id = record_sets[0] if record_sets else None
print(f"\nExample records from record set '@id': {main_record_set_id}")

if main_record_set_id:
    for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
        print(record)
        if i >= 2:
            break

## 3. Data Extraction
Load data from each available record set into Pandas DataFrames for analysis. All references use the `@id` field as per FAIR² conventions.

In [ ]:
# Extract data from each record set
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"DataFrame for RecordSet {record_set_id} loaded: {df.shape[0]} rows, {df.shape[1]} columns.")
    print(f"Columns: {df.columns.tolist()}")

# Show preview of the principal record set
if main_record_set_id:
    print(f"\nPreview of the principal record set DataFrame [{main_record_set_id}]:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

We demonstrate typical data processing tasks:
- Filtering records based on a numeric field
- Normalizing the numeric field
- Grouping data by a demographic attribute

All fields and columns are referenced by their `@id`.

In [ ]:
# Select numeric and group fields by their @id
# Example: Let's search for "GAD-7 score" and "level_of_education" fields by @id
numeric_field_id = None
group_field_id = None

# Search for example field @ids in the main record set
main_df = dataframes[main_record_set_id]
for col in main_df.columns:
    if "gad_7_score" in col.lower():
        numeric_field_id = col
    if "level_of_education" in col.lower():
        group_field_id = col
print(f"Numeric field selected: {numeric_field_id}")
print(f"Group field selected: {group_field_id}")

# Simple filtering: select respondents with GAD-7 score > 10 (moderate anxiety)
threshold = 10
if numeric_field_id and numeric_field_id in main_df:
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize GAD-7 scores
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group analysis by level_of_education
    if group_field_id and group_field_id in filtered_df:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize distributions and relationships – e.g., GAD-7 scores by education level.

Using only @id references in the column selection.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of GAD-7 scores
if numeric_field_id and numeric_field_id in main_df:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of GAD-7 Scores [{numeric_field_id}]")
    plt.xlabel("GAD-7 Score")
    plt.ylabel("Count")
    plt.show()

# Boxplot of GAD-7 by education level
if numeric_field_id and group_field_id and group_field_id in main_df:
    plt.figure(figsize=(10,6))
    sns.boxplot(data=main_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"GAD-7 Score by Education Level")
    plt.xlabel("Education Level (@id)")
    plt.ylabel("GAD-7 Score")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

This notebook demonstrated loading and exploring a FAIR² Croissant dataset using `mlcroissant`, referencing all internal entities by their `@id` for reproducibility and standardization. Records were loaded, fields inspected, example DataFrames constructed, and exploratory analysis performed on mental health indicators such as GAD-7 scores and their relationship to demographic attributes.

For more advanced analysis, consult the full Croissant schema (`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`), and leverage FAIR² standardized entity referencing.